In [ ]:
!pip -q install groq


In [ ]:
#from groq import Groq
from google.colab import userdata

# Securely retrieve API key from Colab Secrets
client = Groq(api_key=userdata.get("GROQ_API_KEY"))

# Test GPT-OSS 120B
response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {
            "role": "user",
            "content": "What is the purpose of a company's annual financial report? Answer in one sentence."
        }
    ],
    temperature=0.0,
    max_completion_tokens=150
)

print("Model:", response.model)
print("Response:", response.choices[0].message.content)

In [ ]:

import json
import pandas as pd

# Load the frozen evaluation dataset
with open("/content/eval_set.jsonl", "r", encoding="utf-8") as f:
    records = [json.loads(line) for line in f if line.strip()]

eval_df = pd.DataFrame(records)

print("Total questions:", len(eval_df))
print("Unique question IDs:", eval_df["question_id"].nunique())

print("\nQuestion categories:")
print(eval_df["question_category"].value_counts())

# Show only question information, not gold answers
display(eval_df[[
    "question_id",
    "question",
    "question_category"
]].head(5))


In [ ]:

import time

MODEL = "openai/gpt-oss-120b"

# Standardized instruction prompt
SYSTEM_PROMPT = """You are a financial question-answering assistant.

Answer the user's financial question accurately and concisely.

If you cannot determine the answer reliably, respond with:
"I don't know."

Do not invent financial figures or unsupported facts.
"""

test_results = []

# Test only the first 3 questions
for _, row in eval_df.head(3).iterrows():

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": row["question"]}
        ],
        temperature=0.0,
        max_completion_tokens=300
    )

    answer = response.choices[0].message.content or ""

    test_results.append({
        "question_id": row["question_id"],
        "system": "A",
        "answer": answer,
        "chunk_ids": []
    })

    print("Question:", row["question"])
    print("Generated answer:", answer)
    print("-" * 60)

    time.sleep(2)

print("Successfully tested:", len(test_results), "questions")


In [ ]:

# Retest the first FinanceBench question

response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": eval_df.iloc[0]["question"]}
    ],
    temperature=0.0,
    reasoning_effort="low",
    max_completion_tokens=800,
    include_reasoning=False
)

print("Question:", eval_df.iloc[0]["question"])
print("Answer:", repr(response.choices[0].message.content))
print("Finish reason:", response.choices[0].finish_reason)
print("Completion tokens:", response.usage.completion_tokens)


# **Following is our model settings**

Setting	Value
LLM	- (GPT-OSS 120B)
API provider	- (GroqCloud)
Temperature -	(0.0)
Reasoning effort	- (low)
Maximum completion tokens	 - (800)
Include reasoning	- (False)
System prompt	- (Same SYSTEM_PROMPT as before)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:

import json
import time
from pathlib import Path
from tqdm.auto import tqdm
from groq import RateLimitError, APIConnectionError, APIStatusError

# Final agreed settings
MODEL = "openai/gpt-oss-120b"
TEMPERATURE = 0.0
REASONING_EFFORT = "low"
MAX_COMPLETION_TOKENS = 800

# Persistent storage in Google Drive
output_dir = Path("/content/drive/MyDrive/FinanceBenchRAG/results")
output_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / "answers_A.jsonl"

# Load successfully completed questions, if any
completed = set()

if output_file.exists():
    with open(output_file, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                item = json.loads(line)
                if item.get("answer", "").strip():
                    completed.add(item["question_id"])

print("Previously completed:", len(completed))

successful = 0
failed = []

# Run System A on all evaluation questions
for _, row in tqdm(eval_df.iterrows(), total=len(eval_df)):

    question_id = row["question_id"]
    question = row["question"]

    if question_id in completed:
        continue

    answer = None

    # Retry temporary API errors
    for attempt in range(5):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {
                        "role": "system",
                        "content": SYSTEM_PROMPT
                    },
                    {
                        "role": "user",
                        "content": question
                    }
                ],
                temperature=TEMPERATURE,
                reasoning_effort=REASONING_EFFORT,
                max_completion_tokens=MAX_COMPLETION_TOKENS,
                include_reasoning=False
            )

            candidate = (
                response.choices[0].message.content or ""
            ).strip()

            if not candidate:
                raise ValueError("Empty model response")

            answer = candidate
            break

        except (RateLimitError, APIConnectionError,
                APIStatusError, ValueError) as e:

            print(f"Retry {attempt + 1} for {question_id}: {e}")

            if attempt < 4:
                time.sleep(min(15 * (2 ** attempt), 120))

    if answer is None:
        failed.append(question_id)
        continue

    # Save immediately after each successful response
    result = {
        "question_id": question_id,
        "system": "A",
        "answer": answer,
        "chunk_ids": []
    }

    with open(output_file, "a", encoding="utf-8") as f:
        f.write(json.dumps(result, ensure_ascii=False) + "\n")

    completed.add(question_id)
    successful += 1

    # Space out requests
    time.sleep(3)

print("\nExperiment finished")
print("New answers saved:", successful)
print("Total completed:", len(completed))
print("Failed questions:", len(failed))
print("Output file:", output_file)

if failed:
    print("Failed IDs:", failed)


In [ ]:
import pandas as pd

answers_df = pd.read_json(
    "/content/drive/MyDrive/FinanceBenchRAG/results/answers_A.jsonl",
    lines=True
)

print("Total answers:", len(answers_df))

display(answers_df.head(10))

In [ ]:
comparison_df = eval_df[
    ["question_id", "question", "question_category"]
].merge(
    answers_df[["question_id", "answer"]],
    on="question_id",
    how="left"
)

pd.set_option("display.max_colwidth", 200)

display(comparison_df.head(20))

In [ ]:
# Count "I don't know" responses
dont_know = answers_df["answer"].str.contains(
    r"i don't know|i do not know|i’m not sure",
    case=False,
    na=False,
    regex=True
)

print("Total questions:", len(answers_df))
print("I don't know responses:", dont_know.sum())
print("Other responses:", (~dont_know).sum())
print("Abstention rate:", f"{dont_know.mean() * 100:.2f}%")

# Display questions where the model said "I don't know"
display(
    comparison_df[
        comparison_df["question_id"].isin(
            answers_df.loc[dont_know, "question_id"]
        )
    ][["question_id", "question", "answer"]]
)

In [ ]:
import json
from pathlib import Path

output_file = Path(
    "/content/drive/MyDrive/FinanceBenchRAG/results/answers_A.jsonl"
)

with open(output_file, "r", encoding="utf-8") as f:
    results = [json.loads(line) for line in f if line.strip()]

expected_ids = set(eval_df["question_id"])
actual_ids = [r["question_id"] for r in results]

assert len(results) == 80
assert len(set(actual_ids)) == 80
assert set(actual_ids) == expected_ids
assert all(r["system"] == "A" for r in results)
assert all(r["chunk_ids"] == [] for r in results)
assert all(isinstance(r["answer"], str) and r["answer"].strip()
           for r in results)

print("All validation checks passed!")
print("Total answers:", len(results))
print("Unique question IDs:", len(set(actual_ids)))
print("Empty answers:", sum(not r["answer"].strip() for r in results))
print("File size:", round(output_file.stat().st_size / 1024, 2), "KB")